# 6-1절 연습 문제 풀이

이 노트북은 6-1절 연습 문제(6-1 ~ 6-3)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch06/06-01_example.ipynb`를 참고한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import random

import numpy as np
import torch
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# [코드 6-4]의 김소월 시 <엄마야 누나야>
poem = '엄마야 누나야 강변 살자 ' \
       '뜰에는 반짝이는 금모래빛 ' \
       '뒷문 밖에는 갈잎의 노래 ' \
       '엄마야 누나야 강변 살자'
print(poem)
print(f'전체 길이: {len(poem)}자')

엄마야 누나야 강변 살자 뜰에는 반짝이는 금모래빛 뒷문 밖에는 갈잎의 노래 엄마야 누나야 강변 살자
전체 길이: 55자


## 연습 문제 6-1

> 다음은 김소월의 시, <엄마야 누나야>를 한 줄로 붙여 쓴 문자열이다. ([코드 6-4])
> 이 문자열의 어휘 사전을 만들어 보자. 단, 다음 조건에 따라 두 종류의 어휘 사전을 만든다.
> - 각 글자(공백 문자 포함)를 토큰으로 하는 어휘 사전
> - 각 어절을 토큰으로 하는 어휘 사전

In [2]:
# 본문 [코드 6-6]과 같은 방식으로 어휘 사전을 만든다
#   set()으로 중복을 제거하고 sorted()로 순서를 고정해 재현성을 확보한다
EOS_TOKEN = '<eos>'

def build_vocab(tokens, use_eos=True):
    vocab = {EOS_TOKEN: 0} if use_eos else {}
    for i, token in enumerate(sorted(set(tokens)), start=len(vocab)):
        vocab[token] = i
    return vocab

# (1) 각 글자를 토큰으로 하는 어휘 사전 — 공백 문자도 하나의 토큰이다
char_tokens = list(poem)
char_vocab = build_vocab(char_tokens)

# (2) 각 어절을 토큰으로 하는 어휘 사전 — split()이 공백을 기준으로 나눈다
word_tokens = poem.split()
word_vocab = build_vocab(word_tokens)

print(f'글자 토큰: 전체 {len(char_tokens)}개, 고유 {len(char_vocab) - 1}개, 어휘 사전 크기 {len(char_vocab)}')
print(f'어절 토큰: 전체 {len(word_tokens)}개, 고유 {len(word_vocab) - 1}개, 어휘 사전 크기 {len(word_vocab)}')
print()
print('글자 어휘 사전:')
print({k: v for k, v in list(char_vocab.items())})
print()
print('어절 어휘 사전:')
print(word_vocab)

글자 토큰: 전체 55개, 고유 27개, 어휘 사전 크기 28
어절 토큰: 전체 15개, 고유 11개, 어휘 사전 크기 12

글자 어휘 사전:
{'<eos>': 0, ' ': 1, '갈': 2, '강': 3, '금': 4, '나': 5, '노': 6, '누': 7, '는': 8, '뒷': 9, '뜰': 10, '래': 11, '마': 12, '모': 13, '문': 14, '밖': 15, '반': 16, '변': 17, '빛': 18, '살': 19, '야': 20, '엄': 21, '에': 22, '의': 23, '이': 24, '잎': 25, '자': 26, '짝': 27}

어절 어휘 사전:
{'<eos>': 0, '갈잎의': 1, '강변': 2, '금모래빛': 3, '노래': 4, '누나야': 5, '뒷문': 6, '뜰에는': 7, '밖에는': 8, '반짝이는': 9, '살자': 10, '엄마야': 11}


### 풀이 해설

실행 결과부터 보자. **글자 어휘 사전이 28개, 어절 어휘 사전이 12개다.**
즉 이 시에서는 **어절 사전이 글자 사전보다 작다.** 흔히 '어절 단위는 어휘 사전이 커진다'고 알려진 것과 반대다.
왜 그럴까? 이 시가 너무 짧기 때문이다. 어절이 전체 15개뿐인데 '엄마야', '누나야', '강변', '살자'가 두 번씩 반복되어
고유 어절이 11개밖에 안 된다. 반면 글자는 55개 중 고유한 것이 27개다.

**이 역전은 데이터가 작을 때만 일어난다.** 텍스트가 커지면 상황이 완전히 뒤집힌다.
한국어에서 쓰이는 글자는 아무리 긴 글이라도 수천 개에서 멈추지만, 어절은 '강변', '강변은', '강변에서', '강변까지'가
모두 다른 토큰이 되어 **글이 길어지는 만큼 끝없이 늘어난다.**
6-3절에서 소설 한 권을 글자 단위로 다룰 때 어휘 사전 크기가 61에 그치는 것과 견주어 보면 차이가 분명하다.

두 단위의 성질을 정리하면 이렇다.

| | 글자 단위 | 어절 단위 |
|---|---|---|
| 어휘 사전 크기 | 텍스트가 길어져도 **거의 늘지 않는다** | 텍스트가 길어지면 **계속 늘어난다** |
| 이 시에서의 크기 | 28 | 12 (데이터가 작아 반대로 나온 경우) |
| 처음 보는 입력 | 글자는 대부분 사전에 있다 | **처음 보는 어절이 자주 나온다**(`<unk>` 필요) |
| 한 토큰이 담는 의미 | 적다 | 많다 |
| 같은 문맥을 보려면 | 더 긴 윈도우가 필요하다 | 짧은 윈도우로 충분하다 |

공백 처리의 차이도 중요하다. **글자 단위에서는 공백(`' '`)이 어휘 사전의 1번 토큰으로 들어가지만,**
어절 단위에서는 공백이 토큰이 아니라 **토큰을 나누는 구분자**라 사전에 없다.
글자 단위로 학습하면 모델이 띄어쓰기까지 함께 학습하게 되는데, 6-3절의 띄어쓰기 모델이 가능한 것도 이 때문이다.

실무에서 글자도 어절도 아닌 **서브워드**(subword) 단위를 쓰는 이유가 이 둘의 절충이다.

참고로 위 코드에서 `sorted(set(...))`를 쓴 것은 본문 [코드 6-6]과 같은 이유다.
`set`은 순회 순서가 보장되지 않으므로 정렬하지 않으면 **실행할 때마다 고유 번호가 달라져 재현이 되지 않는다.**

### 문제 검토

- **적절성: 적합. 6장의 첫 연습 문제로 알맞다.** 본문은 '도레미파솔라시' 일곱 음이라는 작고 깔끔한 예로만 어휘 사전을
  설명했는데, 이 문제가 처음으로 **실제 언어 데이터**에 적용하게 한다. 그리고 두 단위를 함께 만들게 해
  '토큰의 단위는 선택 사항'이라는 것을 몸으로 알게 한다.
- **[검토] 자료를 시로 고른 것이 좋다.** 짧아서 어휘 사전 전체를 눈으로 확인할 수 있고, 반복되는 구절('엄마야 누나야 강변 살자')이
  있어 6-2에서 슬라이딩 윈도우를 만들 때 같은 입력에 같은 정답이 두 번 나오는 상황을 관찰할 수 있다.
- **[검토] '공백 문자 포함'을 명시한 것이 중요하다.** 이 한마디가 없으면 대부분 공백을 빼고 만들 텐데,
  그러면 6-5에서 텍스트를 생성할 때 띄어쓰기 없는 문자열이 나온다. 지문이 이미 잘 챙기고 있다.
- **[검토] 두 사전을 비교해 보라는 요구를 덧붙이면 좋겠다.** 지금은 '만들어 보자'로 끝나 두 사전을 출력하고 넘어가기 쉽다.
  크기를 견주어 보게 하면 토큰 단위 선택의 의미가 드러난다.

**윤문안**

> **6-1** 다음은 김소월의 시, <엄마야 누나야>를 한 줄로 붙여 쓴 문자열이다.
> 이 문자열의 어휘 사전을 만들어 보자. 단, 다음 조건에 따라 두 종류의 어휘 사전을 만들고, 두 사전의 크기를 비교해 보자.
> - 각 글자(공백 문자 포함)를 토큰으로 하는 어휘 사전
> - 각 어절을 토큰으로 하는 어휘 사전

## 연습 문제 6-2

> <엄마야 누나야>를 길이 4의 입력과 길이 1의 정답 쌍으로 재구성해 출력해 보자. [연습 문제 6-1]과 마찬가지로
> 각 글자가 토큰인 경우와 각 어절이 토큰인 경우 각각 출력하며, 시의 마지막은 종료 특수 토큰으로 끝나야 한다.

In [3]:
# 길이 4의 입력 + 길이 1의 정답 = 윈도우 크기 5
WINDOW_SIZE = 5

def make_samples(tokens, window_size=WINDOW_SIZE):
    """토큰 리스트 끝에 <eos>를 붙인 후 슬라이딩 윈도우로 (입력, 정답) 쌍을 만든다."""
    token_list = list(tokens) + [EOS_TOKEN]
    samples = []
    for i in range(len(token_list) - window_size + 1):
        subsequence = token_list[i: i + window_size]
        samples.append((subsequence[:-1], subsequence[-1]))
    return samples

char_samples = make_samples(char_tokens)
word_samples = make_samples(word_tokens)
print(f'글자 토큰 샘플 수: {len(char_samples)}개')
print(f'어절 토큰 샘플 수: {len(word_samples)}개')

글자 토큰 샘플 수: 52개
어절 토큰 샘플 수: 12개


In [4]:
def show(samples, joiner, head=5, tail=3):
    for i, (input_data, label) in enumerate(samples):
        if i < head or i >= len(samples) - tail:
            print(f'  {i:3d}: {joiner.join(input_data)!r} -> {label!r}')
        elif i == head:
            print(f'  ... ({len(samples) - head - tail}개 생략)')

print('[글자 토큰]')
show(char_samples, '')
print()
print('[어절 토큰]')
show(word_samples, ' ')

[글자 토큰]
    0: '엄마야 ' -> '누'
    1: '마야 누' -> '나'
    2: '야 누나' -> '야'
    3: ' 누나야' -> ' '
    4: '누나야 ' -> '강'
  ... (44개 생략)
   49: ' 강변 ' -> '살'
   50: '강변 살' -> '자'
   51: '변 살자' -> '<eos>'

[어절 토큰]
    0: '엄마야 누나야 강변 살자' -> '뜰에는'
    1: '누나야 강변 살자 뜰에는' -> '반짝이는'
    2: '강변 살자 뜰에는 반짝이는' -> '금모래빛'
    3: '살자 뜰에는 반짝이는 금모래빛' -> '뒷문'
    4: '뜰에는 반짝이는 금모래빛 뒷문' -> '밖에는'
  ... (4개 생략)
    9: '갈잎의 노래 엄마야 누나야' -> '강변'
   10: '노래 엄마야 누나야 강변' -> '살자'
   11: '엄마야 누나야 강변 살자' -> '<eos>'


In [5]:
# 반복되는 구절 때문에 같은 입력이 두 번 나오는지 확인한다
from collections import Counter

for name, samples, joiner in [('글자', char_samples, ''), ('어절', word_samples, ' ')]:
    counter = Counter(joiner.join(x) for x, _ in samples)
    duplicated = {k: v for k, v in counter.items() if v > 1}
    print(f'[{name} 토큰] 두 번 이상 나오는 입력: {len(duplicated)}개')
    for key, count in list(duplicated.items())[:5]:
        answers = {y for x, y in samples if joiner.join(x) == key}
        mark = '  <- 정답이 서로 다름!' if len(answers) > 1 else ''
        print(f'    {key!r} x{count} -> 정답 {answers}{mark}')
    print()

[글자 토큰] 두 번 이상 나오는 입력: 10개
    '엄마야 ' x2 -> 정답 {'누'}
    '마야 누' x2 -> 정답 {'나'}
    '야 누나' x2 -> 정답 {'야'}
    ' 누나야' x2 -> 정답 {' '}
    '누나야 ' x2 -> 정답 {'강'}

[어절 토큰] 두 번 이상 나오는 입력: 1개
    '엄마야 누나야 강변 살자' x2 -> 정답 {'<eos>', '뜰에는'}  <- 정답이 서로 다름!



### 풀이 해설

만드는 방법은 본문 [코드 6-2]와 같고, 달라진 것은 **끝에 `<eos>`를 붙인다**는 점 하나다.
`<eos>`를 붙이면 샘플이 하나 늘고, 마지막 샘플의 정답이 `<eos>`가 된다.
이 샘플 하나가 모델에게 '여기가 끝'이라고 알려 주는 유일한 근거다.

**글자 단위와 어절 단위의 샘플 수 차이가 크다.** 글자 단위는 52개, 어절 단위는 12개다.
샘플이 많다는 것은 학습 기회가 많다는 뜻이지만, 입력 4개가 담는 정보는 훨씬 적다.
글자 4개로는 단어 하나도 채우지 못하는 경우가 많다. 앞의 표에서 말한 **'글자 단위는 더 긴 윈도우가 필요하다'**가 여기서 확인된다.

반대로 어절 단위는 입력 4개가 거의 한 행을 담지만, 샘플 수가 너무 적어 학습할 거리가 부족하다.

**반복 구절이 만드는 현상도 눈여겨보자.** 이 시는 '엄마야 누나야 강변 살자'가 처음과 끝에 두 번 나온다.
그래서 같은 입력에 대해 **정답이 서로 다른 샘플 쌍**이 생긴다. 예를 들어 어절 단위에서
`'엄마야 누나야 강변 살자'` 다음에 처음에는 `'뜰에는'`이, 마지막에는 `<eos>`가 온다.

모델은 이런 모순을 만나면 **두 정답의 중간쯤을 배운다.** 본문 p17이 <반짝반짝 작은별>에서
"같은 네 음으로 시작해 뒤로 이어지는 음이 다른 패턴이 여러 개 있어 모델이 혼란을 겪는다"고 한 것이 이 현상이다.
여기서는 시가 짧아 이 충돌이 아주 선명하게 보인다.

### 문제 검토

- **적절성: 적합.** 본문 [코드 6-2](`<eos>` 없음)와 [코드 6-3](`<eos>`, `<pad>` 포함)에서 배운 것을 직접 적용한다.
  6-1에서 만든 두 어휘 사전을 그대로 이어 쓰게 해 문제 사이의 연결도 자연스럽다.
- **[검토] '종료 특수 토큰으로 끝나야 한다'는 조건이 정확하다.** `<pad>`까지 붙이라고 하지 않은 것이 좋다.
  본문 p8이 밝혔듯 `<eos>` 뒤에 `<pad>`가 줄줄이 따라오는 형태는 실제로 쓰지 않는데, 이 문제는 그 함정을 피해 갔다.
- **[검토] 여기서도 한 걸음 더 나아갈 여지가 있다.** 이 시는 첫 행과 마지막 행이 같아 **같은 입력에 정답이 다른 샘플**이 생기는데,
  이는 6-2절 본문이 지적하는 문제를 미리 만나 볼 좋은 기회다. 지금 지문으로는 출력만 보고 지나치기 쉽다.
- **[검토] '길이 4의 입력과 길이 1의 정답'이라는 표현이 명확하다.** 윈도우 크기를 직접 알려 주는 대신
  입력과 정답의 길이로 말해, 독자가 윈도우 크기 5를 스스로 계산하게 한다. 좋은 방식이다.

**윤문안**

> **6-2** <엄마야 누나야>를 길이 4의 입력과 길이 1의 정답 쌍으로 재구성해 출력해 보자. [연습 문제 6-1]과 마찬가지로
> 각 글자가 토큰인 경우와 각 어절이 토큰인 경우 각각 출력하며, 시의 마지막은 종료 특수 토큰으로 끝나야 한다.
> 출력한 결과에서 입력은 같은데 정답이 다른 샘플이 있는지도 찾아보자.

## 연습 문제 6-3

> [연습 문제 6-2]의 입력과 정답 쌍 중 처음과 마지막 쌍을 원-핫 벡터로 인코딩해 출력해 보자.

In [6]:
def encode_pair(input_tokens, label_token, vocab):
    """입력 토큰 목록과 정답 토큰을 원-핫 인코딩한다."""
    input_idx = torch.tensor([vocab[token] for token in input_tokens])
    label_idx = torch.tensor(vocab[label_token])
    # 입력은 모델에 넣을 실수형 텐서로 변환한다
    input_onehot = F.one_hot(input_idx, num_classes=len(vocab)).float()
    label_onehot = F.one_hot(label_idx, num_classes=len(vocab)).float()
    return input_onehot, label_onehot

for name, samples, vocab, joiner in [('글자', char_samples, char_vocab, ''),
                                     ('어절', word_samples, word_vocab, ' ')]:
    print(f'===== {name} 토큰 =====')
    for position, (input_tokens, label_token) in [('처음', samples[0]), ('마지막', samples[-1])]:
        input_onehot, label_onehot = encode_pair(input_tokens, label_token, vocab)
        print(f'[{position} 쌍] {joiner.join(input_tokens)!r} -> {label_token!r}')
        print(f'  입력 텐서 형태: {tuple(input_onehot.shape)}  (S, vocab_size)')
        print(f'  정답 텐서 형태: {tuple(label_onehot.shape)}  (vocab_size,)')
        print(f'  입력 원-핫 벡터에서 1의 위치: {input_onehot.argmax(dim=1).tolist()}')
        print(f'  정답 원-핫 벡터에서 1의 위치: {label_onehot.argmax().item()}')
        print(input_onehot)
        print()

===== 글자 토큰 =====
[처음 쌍] '엄마야 ' -> '누'
  입력 텐서 형태: (4, 28)  (S, vocab_size)
  정답 텐서 형태: (28,)  (vocab_size,)
  입력 원-핫 벡터에서 1의 위치: [21, 12, 20, 1]
  정답 원-핫 벡터에서 1의 위치: 7
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

[마지막 쌍] '변 살자' -> '<eos>'
  입력 텐서 형태: (4, 28)  (S, vocab_size)
  정답 텐서 형태: (28,)  (vocab_size,)
  입력 원-핫 벡터에서 1의 위치: [17, 1, 19, 26]
  정답 원-핫 벡터에서 1의 위치: 0
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0

### 풀이 해설

본문 [코드 6-1]과 같은 방식이다. 토큰을 어휘 사전의 고유 번호로 바꾼 뒤 `F.one_hot()`에 넣고,
모델에 넣을 것이므로 `float()`으로 실수형으로 바꾼다.

출력에서 확인할 것이 세 가지다.

**1. 입력 텐서의 형태가 `(S, vocab_size)`다.** 여기서 `S`는 4(입력 토큰 수), `vocab_size`는 어휘 사전 크기다.
본문 p10의 추가 설명이 말한 대로 `input_size`는 **순차 데이터의 길이가 아니라 어휘 사전의 크기**다.
헷갈리기 쉬운 지점인데, 이 문제를 풀면서 형태를 직접 찍어 보면 분명해진다.
실제 모델에 넣을 때는 여기에 배치 차원을 더해 `(B, S, vocab_size)` 형태가 된다.

**2. 벡터의 길이가 어휘 사전 크기를 그대로 따른다.** 글자 쪽은 28, 어절 쪽은 12다.
6-1에서 확인한 어휘 사전 크기가 그대로 텐서의 두 번째 차원이 된다.
원-핫 벡터는 어느 쪽이든 **1이 하나뿐이고 나머지는 전부 0**이라는 점도 눈에 띈다.
사전이 커질수록 0의 비율이 커지니, 본문 p6이 지적한 "토큰 수가 1만 개라면 요소 하나를 표현하기 위해
길이 1만짜리 벡터가 매번 만들어진다"는 낭비가 어떤 모습인지 작은 규모로 미리 보는 셈이다.

**3. 마지막 쌍의 정답이 `<eos>`이고, 그 원-핫 벡터에서 1의 위치는 0이다.** 어휘 사전을 만들 때
`<eos>`에 0번을 먼저 배정했기 때문이다.

한 가지 덧붙이면, **정답은 실제 학습에서 원-핫으로 만들지 않는다.** 파이토치의 `nn.CrossEntropyLoss`는
정답을 원-핫 벡터가 아니라 **클래스 번호 하나**로 받기 때문이다.
본문 [코드 6-6]의 `MelodyDataset`도 정답은 `tensor(0)`처럼 번호 그대로 반환한다.
이 문제에서 정답까지 원-핫으로 만들어 본 것은 인코딩 과정을 눈으로 확인하기 위한 것이다.

### 문제 검토

- **적절성: 적합.** 6-1(어휘 사전) → 6-2(샘플 재구성) → 6-3(원-핫 인코딩)으로 이어지는 세 문제가
  6-1절 본문의 순서를 그대로 따라간다. 데이터 준비 과정을 처음부터 끝까지 한 번 통과해 보는 구성이라 짜임새가 좋다.
- **[검토] '처음과 마지막 쌍'만 고른 것이 현명하다.** 전부 출력하면 화면이 원-핫 벡터로 뒤덮여 오히려 아무것도 안 보인다.
  게다가 마지막 쌍은 정답이 `<eos>`라 특수 토큰의 인코딩까지 함께 확인된다. 의도된 선택으로 보인다.
- **★ [검토] 정답도 원-핫으로 인코딩해야 하는지가 모호하다.** '입력과 정답 쌍을 원-핫 벡터로 인코딩'이라고 했는데,
  실제 학습에서 **정답은 원-핫으로 만들지 않는다**(`nn.CrossEntropyLoss`가 클래스 번호를 받는다).
  본문 [코드 6-6]의 `MelodyDataset`도 정답은 번호 그대로 내보낸다.
  이 문제를 곧이곧대로 푼 독자가 '정답도 원-핫으로 만들어야 하는구나'라고 잘못 배울 여지가 있다.
  → **연습 문제 뒤에 한 줄 덧붙이거나, 입력만 인코딩하도록 범위를 좁히는 편이 안전하다.**
- **[검토] 텐서 형태를 확인하라는 요구가 있으면 좋겠다.** 이 문제의 실질적 소득은 `(S, vocab_size)` 형태를
  눈으로 확인하는 것인데, '출력해 보자'만으로는 숫자 덩어리를 보고 넘어가기 쉽다.

**윤문안**

> **6-3** [연습 문제 6-2]의 입력과 정답 쌍 중 처음과 마지막 쌍을 원-핫 벡터로 인코딩해 출력하고, 입력 텐서의 형태도 확인해 보자.
> 참고로 실제 학습에서 정답은 원-핫 벡터가 아니라 어휘 사전의 고유 번호 하나로 사용한다. 그 이유는 3장에서 다룬
> 교차 엔트로피 손실 함수의 사용 방법을 떠올려 보면 알 수 있다.